In [4]:
x = 10


def outer():
    x = 20

    def middle():
        global x
        x = 30

        def inner():
            nonlocal x  # Wait, what does this refer to?
            x = 40

        inner()

    middle()
    print(x)


outer()
print(x)

SyntaxError: no binding for nonlocal 'x' found (4236734018.py, line 12)

In [ ]:
def factory(n, data=[]):
    data.append(n)
    return data


print(factory(1))
print(factory(2, [100]))
print(factory(3))

[1]
[100, 2]
[1, 3]


In [6]:
class Counter:
    count = 0

    def __init__(self):
        self.count += 1


c1 = Counter()
c2 = Counter()
print(f"{Counter.count}, {c1.count}, {c2.count}")

0, 1, 1


Here is exactly why += crashes inside a function but works perfectly inside a class instance. It all comes down to how Python translates the += operator in Object-Oriented Programming.

Python unpacks self.count += 1 into this:
self.count = self.count + 1

When Python executes this, it does it in two distinct phases: The Read (Right Side) and The Write (Left Side).

Phase 1: The Read (Right Side)
Python looks at self.count + 1.

It looks inside the instance (c1) to find count. It is empty.

The OOP Magic: Unlike a function, which just throws an error if a local variable is missing, an instance has a fallback. It automatically looks up at its Class (Counter).

It finds Counter.count which is 0.

The math evaluates to 0 + 1 = 1.

Phase 2: The Write (Left Side)
Now Python executes the assignment: self.count = 1.

The Write Rule: When you assign a value to self.attribute, Python always creates or updates that attribute directly inside the instance's local memory.

It does not go back up to the Class to overwrite it. It just creates a brand new local attribute named count attached only to c1.

Why Functions Crash but Classes Don't
Functions use LEGB Scope: Python decides what is local before the code runs. If it sees x += 1, it says, "You are trying to modify a local variable before giving it a base value. Crash."

Classes use Namespace Dictionaries: Classes resolve variables dynamically at runtime by checking a chain of dictionaries. It checks the instance dictionary -> then the class dictionary.

It uses the Class as a reference to do the math, but uses the Instance to save the result.

In [7]:
items = [1, 2, 3]
gen = (x * 2 for x in items)
items.append(4)
print(list(gen))

[2, 4, 6, 8]


In [8]:
result = [y := x + 1 for x in range(3)]
print(y)
print(result)

3
[1, 2, 3]


In [ ]:
x = 10


def outer():
    x = 20

    def middle():
        global x
        x = 30

        def inner():
            x = 40
            print(locals())

        inner()

    middle()
    print(x)


outer()
print(x)

{'x': 40}
20
30


In [11]:
x = 10


def outer():
    x = 20

    def middle():
        global x
        x = 30

        def inner():
            print(x)

        inner()

    middle()
    print(x)


outer()
print(x)

30
20
30


If nonlocal isn't allowed to touch global variables, it should just ignore the global one in middle() and keep walking up the ladder until it finds the x = 20 in outer(), right?

That is exactly how a human brain works. But that is not how Python's compiler works.

Here is the core reason why it crashes instead of giving you outer's x: Python does not skip names. ### The Lexical Search (How Python's Compiler Thinks)

When Python compiles your script (before it even runs the code), it looks at inner() and sees nonlocal x. Here is exactly what the compiler does:

Step 1: It steps out of inner() and looks at the very first enclosing room: middle().

Step 2: It asks, "Hey middle(), do you have a variable named x mapped in your space?"

Step 3: middle() says, "Yes, I do! But I used the global keyword, so my x is a direct pipeline to the Global scope."

Here is exactly where the crash happens.

Instead of saying, "Oh, that's a global pipeline. nonlocal isn't allowed to use that, let me ignore it and go check outer()," Python's compiler says, "I found an x! My search is over! ...Wait, this is bound to a global variable. nonlocal is strictly forbidden from pointing to globals. CRASH."

The "Shield" Effect
When you put global x inside middle(), you didn't just point middle() to the top. You built a solid wall that blocks inner() from ever seeing outer().

Because Python stops searching the exact microsecond it finds a matching name in the chain, it gets trapped by middle's global declaration. It will never, ever reach outer().

How to prove this is true
If you completely remove x from middle(), watch what happens:

In [ ]:
x = 10


def outer():
    x = 20

    def middle():
        # global x is GONE. middle() has no 'x' at all.

        def inner():
            nonlocal x
            x = 40
            print(f"x in inner is : {x}")

        inner()

    print(f"x in middle is : {x}")
    middle()


outer()
print(f"x in last is : {x}")

x in middle is : 20
x in inner is : 40
x in last is : 10


Now, when the compiler checks middle(), it finds absolutely nothing. Only then will it step up to outer(), find x = 20, and perfectly bind to it without crashing!

You didn't misunderstand the concept of the scopes; you just got caught by the compiler's stubborn "stop at the very first match" rule